In [4]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH = f"{DRIVE_BASE}/pubmedqa_filtered.json"
CHECKPOINT_DIR = f"{DRIVE_BASE}/biomistral_lens_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"Corpus loaded: {len(corpus)} samples")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Corpus loaded: 759 samples


In [5]:
MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    use_safetensors=False
)

model = model.to("cuda")
model.eval()

for param in model.parameters():
    param.requires_grad = False

n_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vocab_size = model.config.vocab_size

print(f"Model loaded on GPU: {n_layers} layers, hidden size {hidden_size}")
print(f"Model parameters frozen: {sum(p.requires_grad for p in model.parameters())} trainable")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The repo does no

Model loaded on GPU: 32 layers, hidden size 4096
Model parameters frozen: 0 trainable


In [6]:
class TunedLensTranslator(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.translator = nn.Linear(hidden_size, hidden_size, bias=True)
        nn.init.eye_(self.translator.weight)
        nn.init.zeros_(self.translator.bias)

    def forward(self, hidden_state):
        h = hidden_state.float()
        translated = self.translator(h)
        return translated

translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])

translators = translators.to("cuda")

total_params = sum(p.numel() for p in translators.parameters())
print(f"Tuned lens translators: {n_layers} layers")
print(f"Parameters per translator: {hidden_size * hidden_size + hidden_size:,}")
print(f"Total tuned lens parameters: {total_params:,}")

Tuned lens translators: 32 layers
Parameters per translator: 16,781,312
Total tuned lens parameters: 537,001,984


In [7]:
N_TRAIN = 200

print(f"Collecting hidden states from {N_TRAIN} training samples...")

hook_storage = {"hidden_states": {}}
hooks = []

def make_hook(layer_idx):
    def hook(module, input, output):
        hook_storage["hidden_states"][layer_idx] = output[0].detach().squeeze(0)
    return hook

for i, block in enumerate(model.model.layers):
    hooks.append(block.register_forward_hook(make_hook(i)))

all_hidden_states = {i: [] for i in range(n_layers)}
all_final_logits = []

with torch.no_grad():
    for idx, sample in enumerate(tqdm(corpus[:N_TRAIN])):
        query = sample["query"]
        prompt = f"Question: {query}\nAnswer:"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to("cuda")

        hook_storage["hidden_states"].clear()
        outputs = model(**inputs)

        final_logits = outputs.logits[0, -1, :].float()
        all_final_logits.append(final_logits.cpu())

        for layer_idx in range(n_layers):
            hs = hook_storage["hidden_states"][layer_idx][-1, :]
            all_hidden_states[layer_idx].append(hs.cpu())

for h in hooks:
    h.remove()

final_logits_tensor = torch.stack(all_final_logits)
hidden_tensors = {
    i: torch.stack(all_hidden_states[i]) for i in range(n_layers)
}

print(f"\nTraining data collected:")
print(f"Final logits shape: {final_logits_tensor.shape}")
print(f"Hidden states per layer shape: {hidden_tensors[0].shape}")


100%|██████████| 200/200 [00:10<00:00, 18.21it/s]


Training data collected:
Final logits shape: torch.Size([200, 32000])
Hidden states per layer shape: torch.Size([200, 4096])


In [8]:
translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])

translators = translators.to("cuda")
print("Translators reinitialized from identity.")

Translators reinitialized from identity.


In [9]:
N_EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 32

with torch.no_grad():
    final_probs = torch.softmax(final_logits_tensor, dim=-1)

print(f"Training tuned-lens on {N_TRAIN} samples, {N_EPOCHS} epochs per layer...")
print(f"This trains {n_layers} translators sequentially\n")

layer_losses = {}

for layer_idx in tqdm(range(n_layers), desc="Training layers"):
    hidden = hidden_tensors[layer_idx].to("cuda")
    targets = final_probs.to("cuda")

    translator = translators[layer_idx]
    optimizer = optim.Adam(translator.parameters(), lr=LR)

    best_loss = float('inf')

    for epoch in range(N_EPOCHS):
        epoch_loss = 0
        n_batches = 0

        perm = torch.randperm(N_TRAIN)
        hidden_shuffled = hidden[perm]
        targets_shuffled = targets[perm]

        for batch_start in range(0, N_TRAIN, BATCH_SIZE):
            batch_hidden = hidden_shuffled[batch_start:batch_start+BATCH_SIZE]
            batch_targets = targets_shuffled[batch_start:batch_start+BATCH_SIZE]

            optimizer.zero_grad()

            translated = translator(batch_hidden)
            normed = model.model.norm(translated.half())
            logits = model.lm_head(normed).float()
            probs = torch.softmax(logits, dim=-1)

            kl_loss = torch.nn.functional.kl_div(
                torch.log(probs + 1e-10),
                batch_targets,
                reduction='batchmean'
            )

            kl_loss.backward()
            optimizer.step()

            epoch_loss += kl_loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        if avg_loss < best_loss:
            best_loss = avg_loss

    layer_losses[layer_idx] = best_loss

    if layer_idx % 8 == 0:
        print(f"Layer {layer_idx:2d} trained | best loss: {best_loss:.6f}")

print(f"\nAll {n_layers} layers trained.")
print(f"Mean final loss: {np.mean(list(layer_losses.values())):.6f}")
print(f"Best layer: {min(layer_losses, key=layer_losses.get)} | "
      f"Worst layer: {max(layer_losses, key=layer_losses.get)}")


Training tuned-lens on 200 samples, 50 epochs per layer...
This trains 32 translators sequentially



Training layers:   3%|▎         | 1/32 [00:01<00:36,  1.18s/it]

Layer  0 trained | best loss: 0.694271


Training layers:  28%|██▊       | 9/32 [00:08<00:21,  1.07it/s]

Layer  8 trained | best loss: 0.371674


...

Layer 16 trained | best loss: 0.243948


Training layers:  78%|███████▊  | 25/32 [00:23<00:06,  1.06it/s]

Layer 24 trained | best loss: 4.446389


Training layers: 100%|██████████| 32/32 [00:30<00:00,  1.05it/s]


All 32 layers trained.
Mean final loss: 2.189571
Best layer: 31 | Worst layer: 17


In [10]:
print("saving tuned lens checkpoints")

for layer_idx in range(n_layers):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt")
    torch.save({
        "layer_idx": layer_idx,
        "state_dict": translators[layer_idx].state_dict(),
        "model_id": MODEL_ID,
        "hidden_size": hidden_size,
        "loss": layer_losses[layer_idx]
    }, checkpoint_path)

summary = {
    "model_id": MODEL_ID,
    "n_layers": n_layers,
    "hidden_size": hidden_size,
    "vocab_size": vocab_size,
    "n_train_samples": N_TRAIN,
    "n_epochs": N_EPOCHS,
    "layer_losses": {str(k): v for k, v in layer_losses.items()},
    "mean_loss": float(np.mean(list(layer_losses.values())))
}

with open(os.path.join(CHECKPOINT_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"Checkpoints saved to {CHECKPOINT_DIR}")
print(f"Files: {len(os.listdir(CHECKPOINT_DIR))}")

saving tuned lens checkpoints
Checkpoints saved to /content/drive/MyDrive/medrag/biomistral_lens_checkpoints
Files: 33


In [11]:
print("verifying checkpoint loading")

test_translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])

for layer_idx in range(n_layers):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt")
    checkpoint = torch.load(checkpoint_path)
    test_translators[layer_idx].load_state_dict(checkpoint["state_dict"])

test_translators = test_translators.to("cuda")
test_translators.eval()

sample_hidden = hidden_tensors[0][:1].to("cuda")

with torch.no_grad():
    translated = test_translators[0](sample_hidden)
    normed = model.model.norm(translated.half())
    logits = model.lm_head(normed).float()
    probs = torch.softmax(logits, dim=-1)

print(f"Test translation output shape: {logits.shape}")
print(f"Probabilities sum to 1: {probs.sum().item():.6f}")
print(f"\nTuned-lens checkpoints: confirmed")
print("03_biomistral_tuned_lens_training complete")

print(f"\nAll {n_layers} layers trained.")
print(f"Mean final loss: {np.mean(list(layer_losses.values())):.6f}")
print(f"Best layer: {min(layer_losses, key=layer_losses.get)} | "
      f"Worst layer: {max(layer_losses, key=layer_losses.get)}")

print("\nFull per-layer loss profile:")
print(f"{'Layer':<8} {'Loss':<12} {'Flag'}")
print("-" * 32)
mean_loss = np.mean(list(layer_losses.values()))
for i in range(n_layers):
    flag = " ← HIGH" if layer_losses[i] > mean_loss * 1.5 else ""
    print(f"{i:<8} {layer_losses[i]:<12.6f}{flag}")

verifying checkpoint loading
Test translation output shape: torch.Size([1, 32000])
Probabilities sum to 1: 1.000000

Tuned-lens checkpoints: confirmed
03_biomistral_tuned_lens_training complete

All 32 layers trained.
Mean final loss: 2.189571
Best layer: 31 | Worst layer: 17

Full per-layer loss profile:
Layer    Loss         Flag
--------------------------------
0        0.694271    
1        0.678714    
2        0.679590    
3        0.665861    
4        0.649766    
5        0.543782    
6        0.505830    
7        0.449007    
8        0.371674    
9        0.342166    
10       0.386190    
11       0.352677    
12       0.354575    
13       0.331258    
14       0.310794    
15       0.277374    
16       0.243948    
17       4.558878     ← HIGH
18       4.505313     ← HIGH
19       4.432622     ← HIGH
20       4.402513     ← HIGH
21       4.375028     ← HIGH
22       4.408594     ← HIGH
23       4.430594     ← HIGH
24       4.446389     ← HIGH
25       4.429521     ← HIG